Vamos a realizar el mismo proceso realizado para el corpus de overwatch

In [1]:
import json
from datetime import datetime

INPUT_PATH = "D:\\TFM\\data\\gaming.corpus\\conversations.json"
OUTPUT_PATH = "D:\\TFM\\data\\gaming.corpus\\conversations_truncated.json"
TARGET_N = 300_000  # ajustable

def truncate_conversations_by_recency(input_path, output_path, target_n):
    with open(input_path, "r", encoding="utf-8") as f:
        convs = json.load(f)  # dict {conv_id: {title, timestamp, ...}}

    print(f"Total de posts original: {len(convs)}")

    # Ordenamos por timestamp descendente (más reciente primero)
    sorted_items = sorted(
        convs.items(),
        key=lambda kv: kv[1].get("timestamp", 0),
        reverse=True
    )

    truncated_items = sorted_items[:target_n]
    truncated_dict = dict(truncated_items)

    # Sanity check: rango temporal cubierto por la muestra
    timestamps = [meta.get("timestamp", 0) for _, meta in truncated_items]
    oldest = datetime.utcfromtimestamp(min(timestamps))
    newest = datetime.utcfromtimestamp(max(timestamps))
    print(f"Posts conservados: {len(truncated_dict)}")
    print(f"Rango temporal: {oldest} → {newest}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(truncated_dict, f, ensure_ascii=False)

    print(f"Guardado en: {output_path}")
    return truncated_dict

sampled_conv_ids = set(
    truncate_conversations_by_recency(INPUT_PATH, OUTPUT_PATH, TARGET_N).keys()
)

Total de posts original: 4267583
Posts conservados: 300000
Rango temporal: 2018-04-20 19:27:16 → 2018-10-31 23:59:39
Guardado en: D:\TFM\data\gaming.corpus\conversations_truncated.json


In [2]:
CONV_TRUNCATED_PATH = "D:\\TFM\\data\\gaming.corpus\\conversations_truncated.json"
UTTERANCES_PATH = "D:\\TFM\\data\\gaming.corpus\\utterances.jsonl"
OUTPUT_PATH = "D:\\TFM\\data\\gaming.corpus\\utterances_truncated.jsonl"

def load_sampled_ids(conv_truncated_path):
    with open(conv_truncated_path, "r", encoding="utf-8") as f:
        convs = json.load(f)
    return set(convs.keys())

def truncate_utterances(utterances_path, output_path, sampled_ids):
    n_seen = 0
    n_kept = 0
    n_posts_kept = 0

    with open(utterances_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            n_seen += 1
            utt = json.loads(line)

            if utt.get("root") in sampled_ids:
                fout.write(line)  # escribimos la línea cruda tal cual, sin reserializar
                n_kept += 1
                if utt.get("reply_to") is None:
                    n_posts_kept += 1

            if n_seen % 1_000_000 == 0:
                print(f"Procesadas {n_seen:,} líneas... conservadas: {n_kept:,}")

    print(f"\nTotal líneas procesadas: {n_seen:,}")
    print(f"Total utterances conservadas (posts+comentarios): {n_kept:,}")
    print(f"De ellas, posts: {n_posts_kept:,}")
    print(f"Guardado en: {output_path}")

sampled_ids = load_sampled_ids(CONV_TRUNCATED_PATH)
print(f"Ids de conversación cargados: {len(sampled_ids):,}")

truncate_utterances(UTTERANCES_PATH, OUTPUT_PATH, sampled_ids)

Ids de conversación cargados: 300,000
Procesadas 1,000,000 líneas... conservadas: 52,615
Procesadas 2,000,000 líneas... conservadas: 147,561
Procesadas 3,000,000 líneas... conservadas: 147,561
Procesadas 4,000,000 líneas... conservadas: 208,481
Procesadas 5,000,000 líneas... conservadas: 300,000
Procesadas 6,000,000 líneas... conservadas: 300,000
Procesadas 7,000,000 líneas... conservadas: 300,000
Procesadas 8,000,000 líneas... conservadas: 300,000
Procesadas 9,000,000 líneas... conservadas: 300,000
Procesadas 10,000,000 líneas... conservadas: 300,000
Procesadas 11,000,000 líneas... conservadas: 300,000
Procesadas 12,000,000 líneas... conservadas: 300,000
Procesadas 13,000,000 líneas... conservadas: 300,000
Procesadas 14,000,000 líneas... conservadas: 300,000
Procesadas 15,000,000 líneas... conservadas: 300,000
Procesadas 16,000,000 líneas... conservadas: 300,000
Procesadas 17,000,000 líneas... conservadas: 300,000
Procesadas 18,000,000 líneas... conservadas: 300,000
Procesadas 19,000,

In [3]:
CONV_TRUNCATED_PATH = "D:\\TFM\\data\\gaming.corpus\\conversations_truncated.json"
UTT_TRUNCATED_PATH = "D:\\TFM\\data\\gaming.corpus\\utterances_truncated.jsonl"

# 1. Ids esperados desde conversations_truncated.json
with open(CONV_TRUNCATED_PATH, "r", encoding="utf-8") as f:
    conv_ids = set(json.load(f).keys())

print(f"Ids en conversations_truncated.json: {len(conv_ids):,}")

# 2. Recorremos utterances_truncated.jsonl y separamos posts vs comentarios
post_ids = []
all_roots = set()
n_total = 0

with open(UTT_TRUNCATED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        utt = json.loads(line)
        n_total += 1
        all_roots.add(utt["root"])
        if utt.get("reply_to") is None:
            post_ids.append(utt["id"])

post_ids_set = set(post_ids)

print(f"Total utterances en el fichero truncado: {n_total:,}")
print(f"Posts encontrados (reply_to is None): {len(post_ids):,}")
print(f"Posts únicos (set): {len(post_ids_set):,}")

# 3. Comprobaciones clave
duplicados = len(post_ids) - len(post_ids_set)
faltantes_en_utt = conv_ids - post_ids_set      # ids en conversations pero sin post en utterances
sobrantes_en_utt = post_ids_set - conv_ids      # posts en utterances que no estaban en conversations
roots_fuera_de_conv = all_roots - conv_ids      # cualquier utterance (post o comentario) con root inesperado

print(f"\n--- Resultados de la comprobación ---")
print(f"Duplicados en post_ids: {duplicados}")
print(f"Ids en conversations SIN post correspondiente en utterances: {len(faltantes_en_utt)}")
print(f"Posts en utterances que NO estaban en conversations: {len(sobrantes_en_utt)}")
print(f"Roots (posts+comentarios) fuera del set esperado: {len(roots_fuera_de_conv)}")

coincide_exacto = (conv_ids == post_ids_set)
print(f"\n¿Coinciden exactamente los sets de ids?: {coincide_exacto}")

Ids en conversations_truncated.json: 300,000
Total utterances en el fichero truncado: 4,722,740
Posts encontrados (reply_to is None): 300,000
Posts únicos (set): 300,000

--- Resultados de la comprobación ---
Duplicados en post_ids: 0
Ids en conversations SIN post correspondiente en utterances: 0
Posts en utterances que NO estaban en conversations: 0
Roots (posts+comentarios) fuera del set esperado: 0

¿Coinciden exactamente los sets de ids?: True


In [4]:
import pandas as pd

CONV_TRUNCATED_PATH = "D:\\TFM\\data\\gaming.corpus\\conversations_truncated.json"
UTT_TRUNCATED_PATH = "D:\\TFM\\data\\gaming.corpus\\utterances_truncated.jsonl"
OUTPUT_PATH = "D:\\TFM\\data\\gaming.corpus\\gaming_posts_final.parquet"

def load_conversations(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)  # dict {id: {title, num_comments, domain, timestamp, ...}}

def extract_posts_only(utt_path):
    """Filtra utterances_truncated.jsonl quedándonos solo con los posts (reply_to is None)."""
    posts = []
    n_seen = 0
    with open(utt_path, "r", encoding="utf-8") as f:
        for line in f:
            n_seen += 1
            utt = json.loads(line)
            if utt.get("reply_to") is not None:
                continue  # es comentario, lo descartamos
            posts.append({
                "id": utt["id"],
                "user": utt.get("user"),
                "body_text": utt.get("text", ""),
                "timestamp": utt.get("timestamp"),
                "score": utt.get("meta", {}).get("score"),
            })
    print(f"Líneas totales revisadas: {n_seen:,}")
    print(f"Posts extraídos: {len(posts):,}")
    return posts

# 1. Carga metadata de conversaciones (título, domain, num_comments...)
conv_meta = load_conversations(CONV_TRUNCATED_PATH)

# 2. Extrae solo los posts del jsonl truncado
post_rows = extract_posts_only(UTT_TRUNCATED_PATH)
df_posts = pd.DataFrame(post_rows)

# 3. Construye dataframe de metadata de conversaciones
df_meta = pd.DataFrame([
    {"id": cid, **meta} for cid, meta in conv_meta.items()
])

# 4. Merge por id
df_final = df_posts.merge(df_meta, on="id", how="left", validate="one_to_one")

# 5. Texto final: título + cuerpo
df_final["full_text"] = (
    df_final["title"].fillna("") + " " + df_final["body_text"].fillna("")
).str.strip()

# 6. Sanity checks antes de guardar
print(f"\nFilas finales: {len(df_final):,}")
print(f"Nulos en title: {df_final['title'].isna().sum()}")
print(f"Nulos en body_text: {(df_final['body_text'] == '').sum()} (texto vacío)")
print(f"Filas con full_text vacío tras strip: {(df_final['full_text'] == '').sum()}")

df_final.to_parquet(OUTPUT_PATH, index=False)
print(f"\nGuardado en: {OUTPUT_PATH}")

Líneas totales revisadas: 4,722,740
Posts extraídos: 300,000

Filas finales: 300,000
Nulos en title: 0
Nulos en body_text: 188353 (texto vacío)
Filas con full_text vacío tras strip: 0

Guardado en: D:\TFM\data\gaming.corpus\gaming_posts_final.parquet


In [5]:
import pandas as pd
from datetime import datetime

df = pd.read_parquet("D:\\TFM\\data\\gaming.corpus\\gaming_posts_final.parquet")

# 1. Estructura básica
print(df.shape)
print(df.dtypes)
df.info()

(300000, 15)
id                      str
user                    str
body_text               str
timestamp_x           int64
score                 int64
title                   str
num_comments          int64
domain                  str
timestamp_y           int64
subreddit               str
gilded                int64
gildings             object
stickied               bool
author_flair_text       str
full_text               str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   id                 300000 non-null  str   
 1   user               300000 non-null  str   
 2   body_text          300000 non-null  str   
 3   timestamp_x        300000 non-null  int64 
 4   score              300000 non-null  int64 
 5   title              300000 non-null  str   
 6   num_comments       300000 non-null  int64 
 7   domain          

In [7]:
# 8. Distribución de domain — cuántos son link posts (p.ej. i.redd.it, youtube.com) vs self.gaming (texto puro)
print(df["domain"].value_counts().head(15))
print("\n% posts con domain distinto de self.gaming:",
      (df["domain"] != "self.gaming").mean() * 100)

domain
i.redd.it                         84069
self.gaming                       70173
youtube.com                       30685
reddit.com                        26154
youtu.be                          20493
i.imgur.com                       11168
imgur.com                         10102
v.redd.it                          4694
gfycat.com                         3102
twitch.tv                          1701
clips.twitch.tv                    1527
newsforyou.today                   1490
twitter.com                        1236
m.youtube.com                       903
pcgamerupdatenews.blogspot.com      785
Name: count, dtype: int64

% posts con domain distinto de self.gaming: 76.60900000000001


In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Cargamos el archivo parquet (usamos solo la columna 'full_text' para ahorrar RAM)
df = pd.read_parquet("D:\\TFM\\data\\gaming.corpus\\gaming_posts_final.parquet", columns=['full_text'])

In [9]:
# Inicializamos el vectorizador con optimizaciones
tfidf = TfidfVectorizer(
    stop_words='english',   # Elimina conectores, preposiciones, etc. en inglés
    max_features=10000,     # Nos quedamos solo con las 10,000 palabras más importantes para evitar explotar la RAM
    min_df=5,               # Ignora palabras que aparecen en menos de 5 posts
    max_df=0.7,             # Ignora palabras que aparecen en más del 70% de los posts (demasiado comunes)
    ngram_range=(1, 2)      # Analiza palabras sueltas y frases de dos palabras (ej: "nerf mercy", "competitive queue")
)

# Aplicamos el vectorizador a tu columna 'full_text'
# Esto generará una matriz dispersa (sparse matrix) muy eficiente en memoria
tfidf_matrix = tfidf.fit_transform(df['full_text'].astype(str))

print(f"Matriz TF-IDF creada con dimensiones: {tfidf_matrix.shape}")

Matriz TF-IDF creada con dimensiones: (300000, 10000)


In [10]:
# Obtenemos los nombres de las palabras (features)
feature_names = tfidf.get_feature_names_out()

# Calculamos la media de las puntuaciones TF-IDF para cada palabra (columna) de la matriz
# Usamos mean(axis=0) y lo convertimos a un array plano de numpy
import numpy as np
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).ravel()

# Creamos un DataFrame para visualizar los resultados
importance_df = pd.DataFrame({
    'word': feature_names,
    'tfidf_score': mean_tfidf
})

# Ordenamos de mayor a menor peso TF-IDF
top_words = importance_df.sort_values(by='tfidf_score', ascending=False).head(30)

In [11]:
print(top_words)

                 word  tfidf_score
2131          deleted     0.040471
3311             game     0.022187
7514          removed     0.018146
3580            games     0.014628
8393       standalone     0.012690
3726           gaming     0.011824
6029              new     0.010709
3143         fortnite     0.010013
4837             just     0.009691
5200             like     0.009572
2055             dayz     0.009064
6704             play     0.009026
8967             time     0.007908
9087          trailer     0.007838
921              best     0.007821
2057  dayz standalone     0.006866
9468            video     0.006480
1390          channel     0.006401
3934             good     0.006340
6789          playing     0.006215
7098              ps4     0.005777
6497               pc     0.005664
4986             know     0.005596
3549         gameplay     0.005443
5556              man     0.005359
119              2018     0.005242
3988              got     0.005120
9875             xbo

In [12]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text

# ==========================================
# 1. CARGAR DATOS (solo la columna que usaremos)
# ==========================================
df = pd.read_parquet("D:\\TFM\\data\\gaming.corpus\\gaming_posts_final.parquet", columns=['full_text'])
# ==========================================
# 2. FILTRAR POSTS VACÍOS O ELIMINADOS
# ==========================================
# Eliminamos filas que sean literalmente '[deleted]' o '[removed]'
df = df[~df['full_text'].str.contains(r'^\[deleted\]$|^\[removed\]$', case=False, na=True)]

# ==========================================
# 3. LIMPIEZA DE TEXTO (URLs, caracteres, etc.)
# ==========================================
def clean_text(txt):
    if not isinstance(txt, str):
        return ""
    # Convertir a minúsculas
    txt = txt.lower()
    # Eliminar URLs completas (http, https, www)
    txt = re.sub(r'https?://\S+|www\.\S+', '', txt)
    # Eliminar menciones a usuarios y subreddits (u/usuario, r/subreddit)
    txt = re.sub(r'\b[ru]/\S+', '', txt)
    # Conservar solo letras y espacios (eliminamos números, signos de puntuación extraños)
    txt = re.sub(r'[^a-zA-Z\s]', '', txt)
    # Eliminar espacios múltiples en blanco
    txt = re.sub(r'\s+', ' ', txt).strip()
    return txt

print("Limpiando textos (esto puede tardar alrededor de un minuto)...")
df['clean_text'] = df['full_text'].apply(clean_text)

# Volvemos a descartar filas que hayan quedado vacías tras la limpieza
df = df[df['clean_text'] != ""]

# ==========================================
# 4. ENRIQUECER LA LISTA DE STOP WORDS
# ==========================================
# Partimos de las 318 palabras estándar de sklearn
standard_stops = list(text.ENGLISH_STOP_WORDS)

# Añadimos palabras conversacionales y contracciones que la lista estándar no cubre
conversational_stops = [
    'just', 'like', 'really', 'people', 'think', 'know', 'make', 'want', 've', 'don', 'time', 'one', 
    'even', 'someone', 'something', 'anything', 'everyone', 'still', 'good', 'bad', 'much', 'many', 
    'feel', 'say', 'see', 'look', 'back', 'first', 'new', 'going', 'go', 'use', 'way', 'right', 
    'pretty', 'sure', 'well', 'never', 'always', 'every', 'thing', 'things', 'got', 'did', 'does', 
    'doing', 'made', 'makes', 'saying', 'said', 'says', 'thanks', 'thank', 'guy', 'guys',
    'post', 'posts', 'reddit', 'subreddit', 'thread', 'deleted', 'removed', 'amp', 'im', 'cant', 
    'dont', 'didnt', 'ive', 'id', 'ill', 'youre', 'theyre', 'hes', 'shes'
]

# Añadimos palabras genéricas del "contexto de juego" que no aportan valor temático
domain_stops = [
    'game', 'play', 'playing', 'player', 'players', 'hero', 'heroes', 'match', 
    'matches', 'team', 'teams', 'season', 'competitive', 'comp', 'ranked', 'queue', 'lobby',
    'win', 'lose', 'won', 'lost', 'gameplay', 'video', 'clip', 'main'
]

# Combinamos todas las listas de exclusión
custom_stop_words = set(standard_stops + conversational_stops + domain_stops)

# ==========================================
# 5. CONFIGURAR TF-IDF CON PARÁMETROS AGRESIVOS
# ==========================================
tfidf = TfidfVectorizer(
    stop_words=list(custom_stop_words),
    max_features=10000,
    min_df=15,          # La palabra debe aparecer al menos en 15 posts (evita erratas únicas)
    max_df=0.25,        # ¡CLAVE! Si una palabra aparece en más del 25% de los posts se elimina (adiós a palabras ultra-comunes)
    ngram_range=(1, 2)  # Seguirá detectando bigramas interesantes (ej: "battle pass", "loot box")
)

# ==========================================
# 6. ENTRENAR Y MOSTRAR RESULTADOS
# ==========================================
print("Ajustando matriz TF-IDF...")
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

# Extraer términos y puntuación promedio
feature_names = tfidf.get_feature_names_out()
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).ravel()

importance_df = pd.DataFrame({
    'word': feature_names,
    'tfidf_score': mean_tfidf
})

# Ordenamos e imprimimos el top 30 limpio
top_words = importance_df.sort_values(by='tfidf_score', ascending=False).head(30)
print("\n--- TOP 30 PALABRAS MÁS CARACTERÍSTICAS (LIMPIO) ---")
print(top_words.to_string(index=False))

Limpiando textos (esto puede tardar alrededor de un minuto)...
Ajustando matriz TF-IDF...

--- TOP 30 PALABRAS MÁS CARACTERÍSTICAS (LIMPIO) ---
            word  tfidf_score
           games     0.016588
      standalone     0.013989
          gaming     0.012434
        fortnite     0.011217
         trailer     0.008867
            best     0.008096
              ps     0.007793
         channel     0.007176
            user     0.007009
              pc     0.006121
         fallout     0.006039
            help     0.005612
            xbox     0.005572
           fails     0.005480
 channel trailer     0.005475
       spiderman     0.005360
standalone fails     0.005312
            need     0.005190
          online     0.004763
            dayz     0.004733
            dead     0.004699
         looking     0.004623
          review     0.004553
             war     0.004414
             old     0.004380
            free     0.004316
            love     0.004285
            pubg